# **Prediction Data Prep: Your Turn — Predicting Diabetes**

Rebuild the diabetes data prep **from memory and the slides**. Try not to copy cells from today's demo notebook. Fill in every `# TODO`. Some cells tell you *what* to do but not *which* columns: that decision is yours.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('https://raw.githubusercontent.com/osoliman/DSC110/refs/heads/main/Datasets/nhanes_final.csv')
df.head()

### 1. Filter
We're predicting diabetes in **adults**.

In [ ]:
df = df[df['age'] >= 18].copy()

### 2. Drop columns
Drop anything irrelevant **and** anything that would *not* be available before diabetes is on the provider's radar. Look at the column list and decide.

In [ ]:
df = df.drop(columns=['id', 'hba1c', 'fasting_glucose'])

### 3. Map labels (given)

In [ ]:
gender_map    = {1.0: 'Male', 2.0: 'Female'}
race_map      = {1.0: 'Mexican American', 2.0: 'Other Hispanic', 3.0: 'White',
                 4.0: 'Black', 6.0: 'Asian', 7.0: 'Other'}
education_map = {1.0: 'Less than 9th', 2.0: '9-11th grade', 3.0: 'High school',
                 4.0: 'Some college', 5.0: 'College grad', 9.0: 'Do not know'}

df['gender']         = df['gender'].map(gender_map)
df['race_ethnicity'] = df['race_ethnicity'].map(race_map)
df['education']      = df['education'].map(education_map)

### 4. Missing values
Check first. Then **impute** the numeric columns with their mean and **remove** rows missing education.

In [ ]:
print(df.isna().sum())
for col in ['bmi', 'height_cm', 'systolic_bp']:
    df[col] = df[col].fillna(df[col].mean())
df = df.dropna(subset=['education']).copy()

### 5. Define the target
Keep only clear answers (1 = yes, 2 = no), then recode to 1 and 0. *(Hint: `value_counts()` first. What are 3 and 9?)*

In [ ]:
df = df[df['has_diabetes'].isin([1.0, 2.0])].copy()
df['has_diabetes'] = df['has_diabetes'].map({1.0: 1, 2.0: 0})
df['has_diabetes'].value_counts()

### 6. One-hot encoding

In [ ]:
df = pd.get_dummies(df, columns=['gender', 'race_ethnicity', 'education'], drop_first=True)
df.head()

### 7. Scale numeric columns

In [ ]:
numeric_cols = ['age', 'bmi', 'height_cm', 'systolic_bp']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df[numeric_cols].describe().round(2)

### 8. Train-test split (80/20)

In [ ]:
X = df.drop(columns=['has_diabetes'])
y = df['has_diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

### Reflect (1–2 sentences each)
1. Which columns did you drop in step 2, and why would keeping them be "cheating"?
2. We imputed BMI but *removed* rows missing education. Why treat them differently?
3. About what % of adults in the final data have diabetes? Keep this number. It matters in Week 10.
4. **Stretch:** we computed the mean (step 4) and fit the scaler (step 7) on *all* rows before splitting. Did the test set 'leak' into training? How would you fix it?